In [ ]:
# Kaggle House Prices: Advanced Regression Techniques
# Starter Notebook for COMS Applied AI

# ===============================
# 1. Import libraries
# ===============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# ===============================
# 2. Load the dataset
# ===============================
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

# ===============================
# 3. Exploratory Data Analysis (EDA)
# ===============================
# Distribution of SalePrice
sns.histplot(train["SalePrice"], kde=True)
plt.title("Distribution of House Prices")
plt.show()

# Correlation heatmap (top features)
plt.figure(figsize=(12,8))
corr = train.corr(numeric_only=True)
sns.heatmap(corr["SalePrice"].sort_values(ascending=False).head(15).to_frame(), annot=True, cmap="coolwarm")
plt.title("Top Correlated Features with SalePrice")
plt.show()

# ===============================
# 4. Data Cleaning & Feature Engineering
# ===============================
# Handle missing values
train = train.fillna({"LotFrontage": train["LotFrontage"].median(),
                      "MasVnrArea": 0,
                      "GarageYrBlt": 0})
test = test.fillna({"LotFrontage": test["LotFrontage"].median(),
                    "MasVnrArea": 0,
                    "GarageYrBlt": 0})

# Drop columns with too many NaNs (optional)
train = train.drop(columns=["Alley", "PoolQC", "Fence", "MiscFeature"], errors="ignore")
test = test.drop(columns=["Alley", "PoolQC", "Fence", "MiscFeature"], errors="ignore")

# Log transform skewed target
train["SalePrice"] = np.log1p(train["SalePrice"])

# Combine train+test for consistent encoding
y = train["SalePrice"]
all_data = pd.concat([train.drop("SalePrice", axis=1), test], axis=0)

# One-hot encode categorical features
all_data = pd.get_dummies(all_data)

# Split back
train = all_data.iloc[:len(y), :]
test = all_data.iloc[len(y):, :]

# ===============================
# 5. Model Training & Evaluation
# ===============================
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, random_state=42, verbose=0)
}

results = {}
X_train, X_val, y_train, y_val = train_test_split(train, y, test_size=0.2, random_state=42)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    results[name] = rmse
    print(f"{name}: RMSE = {rmse:.4f}")

# Display results
results_df = pd.DataFrame(list(results.items()), columns=["Model", "RMSE"]).sort_values("RMSE")
print(results_df)

# ===============================
# 6. Feature Importance (example with RandomForest)
# ===============================
feat_imp = pd.Series(models["RandomForest"].feature_importances_, index=train.columns)
feat_imp.nlargest(10).plot(kind='barh')
plt.title("Top 10 Feature Importances (RandomForest)")
plt.show()

# ===============================
# 7. Submission File
# ===============================
best_model = models["XGBoost"]  # pick the lowest RMSE model
best_model.fit(train, y)
preds_test = np.expm1(best_model.predict(test))  # reverse log transform

submission = pd.DataFrame({"Id": pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")["Id"],
                           "SalePrice": preds_test})
submission.to_csv("submission.csv", index=False)
print("Submission file created!")
